# Monotone Subsequence

Author: yunlong(algorithmx)

[back to index](README.md)

# Problem


> Find **all / any / the length of** the longest **strictly increasing / non-decreasing** subsequence of a sequence of integers of length $N$.



# Solution for the case **"all"**

## Simplest solution 



> The simplest solution would be an enumeration algorithm of $O(2^N)$ complexity. Sweeping through the list of integers, one can try either include or not include the element at position `i`, and test the monotonicity of all the $2^N$ subsequences. 

> A little bit thoughts lead to the following recursive implementation, 
where the indices for the longest monotonic subsequence of `L[1:i]` 
is constructed recursively by `mono(i,L)`. Notice that for `i=1`, 
we return both the empty index list and the full index list (which is `[1]`) 
to account for the cases where the longest subsequence excludes `L[1]`. 
At position `i`, `L[i]` can either be included or excluded, 
this leads to `vcat(A,B)` where `A` is the list of subsequence 
indices ended by `i` and `B` is those without `i`. 
We also trim off the short subsequences by a length cutoff `minlen` 
before return.


In [1]:
function mono(i::Int, L::Vector{Int})
    if i==1
        return [Int[],Int[1,]]
    else
        M = mono(i-1,L)
        M = vcat([Int[m...,i] for m in M if length(m)==0 || L[m[end]]<=L[i]],  M)
        minlen = maximum(length.(M)) - ((length(L)-i))
        return filter(x->length(x)>=minlen, M)
    end
end

function longest_v1(L::Vector{Int})
    indices = mono(length(L), L)
    len = length.(indices)
    return indices[len.==maximum(len)]
end

longest_v1 (generic function with 1 method)

## Online algorithm 



> The above implementation can be addapted to an online version, 
which keeps a list of monotonic subsequences of `L[1:i-1]` (not necessarily longest) and waits for the integer `L[i]` as the input. 
Since we need to perform a large number of search and insert  operations each time a new integer arrives, an effecient data structure is necessary. 

> The recursive algorithm in the previous section hints a tree data structure for the online algorithm. Suppose that we have a tree to store all the monotonic subsequences. When a new integer arrives, we need to iterate through the entire tree to find suitable places for that integer. 



In [2]:
# minimal tree data structure
mutable struct Tree{T <: Any}
    D::T
    C::Vector{Tree{T}}
    function Tree(data::S) where S
        X = new{S}()
        X.D = data
        X.C = Tree{S}[]
        return X
    end
end

depth(T::Tree) = length(T.C)==0 ? 0 : maximum([depth(c) for c in T.C]) + 1

function dfs_collect_n(i::Int, n::Int, T::Tree{Int})
    if length(T.C)==0
        return i==n ? [[T.D,],] : []
    else
        @assert i<n
        return [[T.D,x...]  
                    for c in T.C 
                        for x in dfs_collect_n(i+1, n, c)]
    end
end

function entree!(TR::Tree{Int}, i::Int, L::Vector{Int})
    Li = L[i]
    if TR.D > 0 && L[TR.D] > Li
        return
    else
        P = 0
        for j in 1:length(TR.C)
            if Li >= L[TR.C[j].D]
                entree!(TR.C[j], i, L)
                P = j
            else
                break
            end
        end
        TR.C = insert!(TR.C, P+1, Tree(i)) 
        #vcat(TR.C[1:P], Tree{Int}[,], TR.C[(P+1):length(TR.C)])
        return
    end
end

function longest_v2(L::Vector{Int})
    # init
    T = Tree(-1)
    for i in 1:length(L)  entree!(T, i, L)  end
    # DFS to collect the longest non-descending subseq
    map(x->x[2:end], dfs_collect_n(0,depth(T),T))
end

longest_v2 (generic function with 1 method)

## Graph equivalent 

>Define the adjacent matrix  
```` M[i,j] = L[i]<=L[j] && i < j```` 
```
adj(L) = [(i<j && L[i]<=L[j]) for i=1:length(L), j=1:length(L)]
```
>of a graph with $N$ nodes labelled by $V_i \; (i=1...N)$. 
Then we have a directed acyclic graph (DAG). 
A monotonic subsequence `L[i], L[j], L[k]` implies directed edges 
from $V_i$ to $V_j$ and then to $V_k$. The longest monotonic 
subsequence is equivalent to the longest path in the DAG. 

>The adjacent matrix of a DAG must be nilpotent. [1] Therefore, we can generate all the monotonic subsequences by computing $M^k$ for $k=1,2,...N$ until all entries vanishes. This is an algorithm at worst case $O(N^4)$.  

> Consider the following example. 

In [3]:
using SparseArrays

@inline conn(i,j,A,B) = findall(A[i,:].*B[:,j].!=0)

@inline adj(L) = Int[(i<j && L[i]<=L[j])  
                     for i=1:length(L), j=1:length(L)]

function longest_v3(A::Vector{Int})
    # case o A = [1]
    length(A)==1  &&  return [Int[1,]]
    M = sparse(adj(A))
    Mn = SparseMatrixCSC{Int64, Int64}[M,]
    while sum(Mn[end])>0
        push!(Mn, Mn[end]*M)
    end
    # case of A = [3,2,1]
    length(Mn)==1  &&  return [Int[k,] for k=1:length(A)]
    # case of longest length > 1
    R,C,V = findnz(Mn[end-1])
    subseqs = Vector{Int}[Int[r,c,] for (r,c) in zip(R,C)]
    for i = length(Mn)-2:-1:1
        subseqs1 = []
        for seq in subseqs
            for p in conn(seq[1],seq[2],Mn[i],M)
                push!(subseqs1, Int[seq[1],p,seq[2:end]...])
            end
        end
        subseqs = subseqs1
    end
    subseqs
end

longest_v3 (generic function with 1 method)

## Conclusion (after running benchmark)
> + The graph version `longest_v3` is by far the fastest implementation.
 
> + The tree data structure in the online version `longest_v2` is optimizable. 

## Verification and benchmark

In [4]:
# verification
for i=200
    A = rand(1:10000,rand(2:100))
    @assert sort(longest_v1(A))==sort(longest_v2(A))==sort(longest_v3(A))
end

In [5]:
using BenchmarkTools

### Benchmark group 1

In [10]:
b1 = @benchmarkable longest_v1(A) setup=(
        A=rand(1:100,rand(10:50)))
tune!(b1)
run(b1)

BenchmarkTools.Trial: 801 samples with 1 evaluation.
 Range (min … max):  21.747 μs … 213.710 ms  ┊ GC (min … max): 0.00% … 8.99%
 Time  (median):      1.205 ms               ┊ GC (median):    0.00%
 Time  (mean ± σ):    6.257 ms ±  16.231 ms  ┊ GC (mean ± σ):  8.09% ± 9.35%

  █▄▃▂▂▂ ▁ ▁                                                    
  ██████▇███▇▇▇▆█▇▅▅▆▆▅▅▁▅▅▄▄▆▄▄▄▅▄▅▄▄▁▁▁▁▁▁▁▁▁▄▁▁▅▁▁▁▁▁▁▁▁▄▄▄ ▇
  21.7 μs       Histogram: log(frequency) by time      76.7 ms <

 Memory estimate: 9.08 KiB, allocs estimate: 175.

In [11]:
b2 = @benchmarkable longest_v2(A) setup=(
        A=rand(1:100,rand(10:50)))
tune!(b2)
run(b2)

BenchmarkTools.Trial: 580 samples with 1 evaluation.
 Range (min … max):  19.579 μs … 332.348 ms  ┊ GC (min … max):  0.00% … 29.62%
 Time  (median):      1.799 ms               ┊ GC (median):     0.00%
 Time  (mean ± σ):    8.651 ms ±  19.651 ms  ┊ GC (mean ± σ):  14.72% ± 12.04%

  █▄▃▃▁▁ ▁                                                      
  ███████████▇▇▇▆█▆▅█▅▅▅▆▇▄▅▅▅▅▄▄▅▁▁▄▅▅▅▅▄▄▁▁▅▁▅▄▁▅▁▁▁▁▁▄▄▁▁▁▄ ▇
  19.6 μs       Histogram: log(frequency) by time      75.2 ms <

 Memory estimate: 13.48 KiB, allocs estimate: 303.

In [12]:
b3 = @benchmarkable longest_v3(A) setup=(
        A=rand(1:100,rand(10:50)))
tune!(b3)
run(b3)

BenchmarkTools.Trial: 9610 samples with 5 evaluations.
 Range (min … max):    3.500 μs …   2.897 ms  ┊ GC (min … max): 0.00% … 15.43%
 Time  (median):      63.519 μs               ┊ GC (median):    0.00%
 Time  (mean ± σ):   103.312 μs ± 134.287 μs  ┊ GC (mean ± σ):  7.60% ±  9.88%

  ▄█▆▄▂▁                                                         
  ███████▇▇▆▅▄▄▃▃▃▃▂▂▂▂▂▂▂▂▁▁▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁ ▂
  3.5 μs           Histogram: frequency by time          686 μs <

 Memory estimate: 4.86 KiB, allocs estimate: 46.

### Benchmark group 2

In [13]:
b1 = @benchmarkable longest_v1(A) setup=(
        A=rand(1:1000,rand(10:50)))
tune!(b1)
run(b1)

BenchmarkTools.Trial: 1014 samples with 1 evaluation.
 Range (min … max):  21.873 μs … 111.147 ms  ┊ GC (min … max): 0.00% … 6.91%
 Time  (median):      1.135 ms               ┊ GC (median):    0.00%
 Time  (mean ± σ):    4.926 ms ±   9.490 ms  ┊ GC (mean ± σ):  5.38% ± 8.01%

  █▅▃▃▃▂▂▁▁▁  ▁                                                 
  ██████████████▆█▇▇▇▅▆▅▅▅▆▄▅▆▆▅▅▆▅▅▅▁▄▅▅▅▅▅▄▁▄▅▁▁▄▄▁▄▅▄▁▁▁▁▁▄ ▇
  21.9 μs       Histogram: log(frequency) by time      47.7 ms <

 Memory estimate: 8.84 KiB, allocs estimate: 175.

In [14]:
b2 = @benchmarkable longest_v2(A) setup=(
        A=rand(1:1000,rand(10:50)))
tune!(b2)
run(b2)

BenchmarkTools.Trial: 442 samples with 1 evaluation.
 Range (min … max):  29.200 μs … 312.370 ms  ┊ GC (min … max):  0.00% … 27.67%
 Time  (median):      1.821 ms               ┊ GC (median):     0.00%
 Time  (mean ± σ):   11.312 ms ±  29.240 ms  ┊ GC (mean ± σ):  20.62% ± 13.24%

  █▄▃▂▁▁                                                        
  ██████▇█▆▆▇▇▆▅▅▆▆▆▄▄▄▄▆▄▄▁▅▁▁▁▁▄▄▁▅▁▁▁▁▁▁▁▁▅▁▁▁▁▁▄▁▁▁▄▄▄▁▁▁▅ ▆
  29.2 μs       Histogram: log(frequency) by time       140 ms <

 Memory estimate: 18.77 KiB, allocs estimate: 424.

In [15]:
b3 = @benchmarkable longest_v3(A) setup=(
        A=rand(1:1000,rand(10:50)))
tune!(b3)
run(b3)

BenchmarkTools.Trial: 8921 samples with 5 evaluations.
 Range (min … max):    1.798 μs …   2.212 ms  ┊ GC (min … max): 0.00% … 0.00%
 Time  (median):      66.751 μs               ┊ GC (median):    0.00%
 Time  (mean ± σ):   111.206 μs ± 146.909 μs  ┊ GC (mean ± σ):  7.64% ± 9.84%

  ▂█▅▃▂▁                                                         
  ███████▇▆▅▅▄▄▃▃▃▂▂▂▂▂▂▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁ ▂
  1.8 μs           Histogram: frequency by time          734 μs <

 Memory estimate: 3.28 KiB, allocs estimate: 24.

### Benchmark group 3

In [16]:
b1 = @benchmarkable longest_v1(A) setup=(
        A=rand(1:1000,rand(50:100)))
tune!(b1)
run(b1)

BenchmarkTools.Trial: 10 samples with 1 evaluation.
 Range (min … max):    9.076 ms …    3.054 s  ┊ GC (min … max):  0.00% … 14.31%
 Time  (median):     208.160 ms               ┊ GC (median):    14.93%
 Time  (mean ± σ):   565.937 ms ± 942.035 ms  ┊ GC (mean ± σ):  17.18% ± 16.80%

  █                                                              
  █▁▁▆▆▁▁▆▁▁▁▆▁▁▁▁▁▁▁▁▁▁▆▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▆ ▁
  9.08 ms          Histogram: frequency by time          3.05 s <

 Memory estimate: 6.19 MiB, allocs estimate: 51596.

In [17]:
b2 = @benchmarkable longest_v2(A) setup=(
        A=rand(1:1000,rand(50:100)))
tune!(b2)
run(b2)

BenchmarkTools.Trial: 10 samples with 1 evaluation.
 Range (min … max):   12.149 ms … 3.860 s  ┊ GC (min … max):  0.00% … 33.41%
 Time  (median):     581.151 ms            ┊ GC (median):    27.80%
 Time  (mean ± σ):   854.749 ms ± 1.139 s  ┊ GC (mean ± σ):  34.64% ± 13.47%

  █ █   ▁    ▁▁▁     ▁                                     ▁  
  █▁█▁▁▁█▁▁▁▁███▁▁▁▁▁█▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁█ ▁
  12.1 ms        Histogram: frequency by time         3.86 s <

 Memory estimate: 9.94 MiB, allocs estimate: 235853.

In [18]:
b3 = @benchmarkable longest_v3(A) setup=(
        A=rand(1:1000,rand(50:100)))
tune!(b3)
run(b3)

BenchmarkTools.Trial: 5609 samples with 1 evaluation.
 Range (min … max):   89.073 μs … 74.682 ms  ┊ GC (min … max): 0.00% …  4.81%
 Time  (median):     514.150 μs              ┊ GC (median):    0.00%
 Time  (mean ± σ):   886.637 μs ±  1.852 ms  ┊ GC (mean ± σ):  6.99% ± 11.27%

  ▂██▅▄▂                                                        
  ██████▇▆▅▅▄▄▃▃▃▃▃▃▃▃▃▂▃▂▂▂▂▂▂▂▂▂▂▂▂▂▂▂▂▂▂▂▂▂▂▂▂▂▂▂▁▂▂▁▂▂▁▂▂▂ ▃
  89.1 μs         Histogram: frequency by time         6.31 ms <

 Memory estimate: 222.64 KiB, allocs estimate: 279.